# Clean up web-search blocklist Function resources

Delete only resources recorded by this lab: its RBAC assignments on the **existing** Foundry account, its dedicated resource group, and its Entra app registrations. The existing Foundry account and model deployment remain.

Use the same lab working directory, tenant, and subscription as the deployment notebook. This notebook can recover lab identity IDs and Foundry role assignments even if deployment failed before outputs were saved. Do not delete the lab managed identities first: their object IDs are needed to locate any remaining assignments. APIM may remain soft-deleted after resource-group deletion; purge it separately only if you need to reuse its name.


In [ ]:
import json
from pathlib import Path
from lab_helpers import az

state_path = Path('.lab-state.json')
assert state_path.exists(), 'No saved lab state. Identify the lab resources explicitly before cleanup.'
state = json.loads(state_path.read_text())
account = az('account', 'show')
assert account['tenantId'] == state['tenant_id']
assert account['id'] == state['subscription_id']
assert state.get('lab_name') == 'ai-foundry-web-search-blocklist-function', 'State belongs to another lab.'
resource_group = state['resource_group']
group_exists = az('group', 'exists', '--name', resource_group)
outputs = state.get('outputs', {})
assignment_ids = set(outputs.get('foundryRoleAssignmentIds', []))
if group_exists:
    group = az('group', 'show', '--name', resource_group)
    assert (group.get('tags') or {}).get('ai-gateway-lab') == state['lab_name'], 'Group ownership tag is missing; identify resources before cleanup.'
    resources = az('resource', 'list', '--resource-group', resource_group)
    principal_ids = set()
    for resource in resources:
        if ((resource['type'].lower() == 'microsoft.apimanagement/service' and resource['name'].startswith('apim-wsbf-'))
                or (resource['type'].lower() == 'microsoft.web/sites' and resource['name'].startswith('func-wsbf-'))):
            detail = az('resource', 'show', '--ids', resource['id'])
            principal_id = (detail.get('identity') or {}).get('principalId')
            if principal_id:
                principal_ids.add(principal_id)
    if principal_ids:
        assignments = az('role', 'assignment', 'list', '--scope', state['foundry_id'],
                         '--subscription', state['foundry_subscription_id'], '--include-inherited')
        assignment_ids.update(item['id'] for item in assignments
                              if item['principalId'] in principal_ids
                              and item['scope'].lower() == state['foundry_id'].lower()
                              and item['roleDefinitionId'].endswith('/5e0bd9bd-7b93-4f28-af87-19fc36ad61bd'))
outputs['foundryRoleAssignmentIds'] = sorted(assignment_ids)
state['outputs'] = outputs
state_path.write_text(json.dumps(state, indent=2) + '\n')
app_ids = [state[f'{name}_app']['appId'] for name in ('gateway', 'notebook', 'function') if f'{name}_app' in state]
print('Resource group:', resource_group, '(exists:', group_exists, ')')
print('App registrations:', app_ids)
print('Foundry role assignments:', sorted(assignment_ids))

external_api_id = None
if state.get('apim_reused'):
    external_api_id = outputs.get('apiId') or state['apim_service_id'].rstrip('/') + '/apis/web-search-blocklist-function'
    assert external_api_id.lower() == (state['apim_service_id'].rstrip('/') + '/apis/web-search-blocklist-function').lower()
    print('Lab API to remove from the retained shared APIM:', external_api_id)
    print('Existing APIM and its pre-existing Foundry role assignments will be retained.')


In [ ]:
confirmation = input(f'Type {resource_group} to delete these lab resources: ')
assert confirmation == resource_group, 'Cleanup cancelled.'
if external_api_id:
    parts = state['apim_service_id'].strip('/').split('/')
    apis = az('apim', 'api', 'list', '--service-name', parts[7], '--resource-group', parts[3], '--subscription', parts[1])
    if any(item['id'].lower() == external_api_id.lower() for item in apis):
        az('rest', '--method', 'DELETE', '--url', f'https://management.azure.com{external_api_id}?api-version=2024-05-01')
for assignment_id in sorted(assignment_ids):
    az('role', 'assignment', 'delete', '--ids', assignment_id, '--subscription', state['foundry_subscription_id'])
if group_exists:
    az('group', 'delete', '--name', resource_group, '--yes')
for client_id in app_ids:
    if az('ad', 'app', 'list', '--filter', f"appId eq '{client_id}'"):
        az('ad', 'app', 'delete', '--id', client_id)
for path in (Path('params.json'), Path('function.zip'), state_path):
    path.unlink(missing_ok=True)
print('Lab resources, role assignments, registrations, and generated deployment files removed.')
